# geoje_mean: Diffusion-TS forecasting

Sequence length is fixed to 128. Run cells from top to bottom. Training weights are shared across tasks.


In [ ]:
from pathlib import Path
import json, subprocess, sys, threading
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while ROOT.name != "Diff-ts" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert ROOT.name == "Diff-ts", "Open this notebook from inside the Diff-ts repository."
NAME = "geoje_mean"
TRUTH_NAME = "geoje_mean"
CONFIG = ROOT / "Config/geoje/geoje_mean_128.yaml"
SEQ_LEN = 128
GPU = 0
MILESTONE = 10
TASK = "forecasting"
TRAIN_PROPORTION = 0.9
SEEDS = [0, 42, 123]
GPU_BY_SEED = {0: 1, 42: 2, 123: 3}
RUN_TRAINING = False  # Uncond starts training when the training cell runs.
RUN_SAMPLING = False  # Set True after the requested checkpoint exists.
print("repository:", ROOT)
print("config:", CONFIG)

def run_parallel_with_live_output(commands_by_seed, phase):
    """Run seed processes concurrently and stream prefixed output to Jupyter."""
    processes = {}
    threads = []

    def stream(seed, gpu, process):
        prefix = f"[{phase} | seed {seed} | GPU {gpu}]"
        for line in iter(process.stdout.readline, ""):
            print(prefix, line.rstrip(), flush=True)
        process.stdout.close()

    for seed, command in commands_by_seed.items():
        gpu = GPU_BY_SEED[seed]
        print(f"[{phase} | seed {seed} | GPU {gpu}] START", flush=True)
        process = subprocess.Popen(
            command,
            cwd=ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        processes[seed] = process
        thread = threading.Thread(target=stream, args=(seed, gpu, process), daemon=True)
        thread.start(); threads.append(thread)

    return_codes = {seed: process.wait() for seed, process in processes.items()}
    for thread in threads: thread.join()
    for seed, code in return_codes.items():
        status = "DONE" if code == 0 else f"FAILED (exit={code})"
        print(f"[{phase} | seed {seed} | GPU {GPU_BY_SEED[seed]}] {status}", flush=True)
    if any(code != 0 for code in return_codes.values()):
        raise RuntimeError(f"{phase} failed: {return_codes}")
    return return_codes


## 1. Train the shared unconditional diffusion model

This is the README `Training` command. Set `RUN_TRAINING=True` above to execute it.


In [ ]:
def training_command(seed=None):
    gpu = GPU_BY_SEED[seed] if seed is not None else GPU
    cmd = [sys.executable, "-u", "main.py", "--name", NAME, "--config_file", str(CONFIG), "--gpu", str(gpu), "--task", TASK, "--train"]
    if seed is not None:
        cmd += ["--seed", str(seed), "--run_id", f"seed_{seed}"]
    cmd += ["dataloader.train_dataset.params.proportion", str(TRAIN_PROPORTION)]
    return cmd

training_seeds = SEEDS if TASK == "uncond" else [None]
train_commands = [training_command(seed) for seed in training_seeds]
for train_cmd in train_commands: print(" ".join(train_cmd))
if RUN_TRAINING:
    if TASK == "uncond":
        run_parallel_with_live_output(dict(zip(SEEDS, train_commands)), "TRAIN")
        for seed in SEEDS:
            checkpoint_dir = ROOT / "artifacts" / "uncond" / NAME / f"seed_{seed}" / "checkpoints_128"
            final_checkpoint = checkpoint_dir / f"checkpoint-{MILESTONE}.pt"
            assert final_checkpoint.exists(), f"Final checkpoint missing: {final_checkpoint}"
            for milestone in range(1, MILESTONE):
                old_checkpoint = checkpoint_dir / f"checkpoint-{milestone}.pt"
                if old_checkpoint.exists(): old_checkpoint.unlink()
            print(f"seed {seed}: retained checkpoint-{MILESTONE}.pt and removed checkpoints 1-{MILESTONE - 1}")
    else:
        subprocess.run(train_commands[0], cwd=ROOT, check=True)


## 2. Forecast 64 steps

This is the README `Forecasting` command. Repeated diffusion draws form a predictive ensemble.


In [ ]:
PRED_LEN = 64
ENSEMBLE_SIZE = 20
sample_cmd = [sys.executable, "main.py", "--name", NAME, "--config_file", str(CONFIG), "--gpu", str(GPU), "--sample", "1", "--milestone", str(MILESTONE), "--mode", "predict", "--pred_len", str(PRED_LEN)]
print(" ".join(sample_cmd))
artifact = ROOT / "artifacts" / "forecasting" / NAME
ensemble_path = artifact / f"ddpm_predict_{NAME}_{SEQ_LEN}_ensemble.npy"
if RUN_SAMPLING:
    draws = []
    for draw in range(ENSEMBLE_SIZE):
        draw_cmd = sample_cmd[:2] + ["--seed", str(12345 + draw)] + sample_cmd[2:]
        subprocess.run(draw_cmd, cwd=ROOT, check=True)
        draws.append(np.load(artifact / f"ddpm_predict_{NAME}_{SEQ_LEN}.npy"))
    np.save(ensemble_path, np.stack(draws))


## 3. Forecast metrics, predictive interval, and visualization


In [ ]:
truth_file = f"sine_ground_truth_{SEQ_LEN}_test.npy" if TRUTH_NAME == "sine" else f"{TRUTH_NAME}_norm_truth_{SEQ_LEN}_test.npy"
truth_path = artifact / "samples" / truth_file
assert ensemble_path.exists(), f"Run ensemble sampling first: {ensemble_path}"
assert truth_path.exists(), f"Missing test truth: {truth_path}"
ensemble, truth = np.load(ensemble_path), np.load(truth_path)
n = min(ensemble.shape[1], len(truth)); ensemble, truth = ensemble[:, :n, -PRED_LEN:], truth[:n, -PRED_LEN:]
mean_forecast = ensemble.mean(axis=0); lower, upper = np.quantile(ensemble, [0.05, 0.95], axis=0)
error = mean_forecast - truth
coverage = np.mean((truth >= lower) & (truth <= upper))
print({"MAE": float(np.mean(np.abs(error))), "RMSE": float(np.sqrt(np.mean(error**2))),
       "90%_coverage": float(coverage), "mean_interval_width": float(np.mean(upper-lower))})
i, feature = 0, 0; x = np.arange(PRED_LEN)
plt.figure(figsize=(13, 4)); plt.plot(x, truth[i, :, feature], label="truth")
plt.plot(x, mean_forecast[i, :, feature], label="forecast")
plt.fill_between(x, lower[i, :, feature], upper[i, :, feature], alpha=.25, label="90% predictive interval")
plt.legend(); plt.tight_layout()
